In [1]:
import logging
import ndjson
from pickhardtpayments.pickhardtpayments import *
from pickhardtpayments.pickhardtpayments.Payment import Payment

In [2]:
from json import JSONEncoder

# subclass JSONEncoder
class PaymentEncoder(JSONEncoder):
    def default(self, o):
        return o.__dict__

In [3]:
# *** Setup ***
# graph = ChannelGraph("./pickhardtpayments/channels.sample.json")
graph = ChannelGraph("./pickhardtpayments/listchannels20220412.json")

def only_channels_with_return_channels(_graph: ChannelGraph):
    channels_with_no_return_channel = []
    for edge in _graph.network.edges:
        if not _graph.network.has_edge(edge[1], edge[0]):
            channels_with_no_return_channel.append(edge)

    for edge in channels_with_no_return_channel:
        _graph.network.remove_edge(edge[0], edge[1], edge[2])

    if len(channels_with_no_return_channel) == 0:
        logging.debug("channel graph only had channels in both directions.")
    else:
        logging.debug("channel graph had unannounced channels.")
    return _graph

graph = only_channels_with_return_channels(graph)
uncertainty_network = UncertaintyNetwork(graph)
oracle_lightning_network = OracleLightningNetwork(graph)

In [5]:
# test of liquidity adjustment
set_liquidity = 100000000
import random
chan = random.sample(list(oracle_lightning_network.network.edges), min(1000, len(oracle_lightning_network.network.edges)))
liquidity_test = True
for chan in random.sample(list(oracle_lightning_network.network.edges), 5):
    c = oracle_lightning_network.get_channel(chan[0], chan[1], chan[2])
    if c.actual_liquidity != set_liquidity:
        liquidity_test = False
        print(f"liquidity not {set_liquidity:,}")
if liquidity_test:
    print(f"liquidity in sample is {set_liquidity:,}")

liquidity not 100,000,000
liquidity not 100,000,000
liquidity not 100,000,000
liquidity not 100,000,000
liquidity not 100,000,000


In [ ]:
sim_session = SyncSimulatedPaymentSession(oracle_lightning_network, uncertainty_network, prune_network=False)
print("Setup finished")

In [ ]:
def create_payment_set(_uncertainty_network, _number_of_payments, amount) -> list[dict]:
    if (len(_uncertainty_network.network.nodes())) < 3:
        logging.warning("graph has less than two nodes")
        exit(-1)
    _payments = []
    while len(_payments) < _number_of_payments:
        # casting _channel_graph to list to avoid deprecation warning for python 3.9
        _random_nodes = random.sample(list(_uncertainty_network.network.nodes), 2)
        # additional check for existence; doing it here avoids the check in each round, improving runtime
        src_exists = _random_nodes[0] in _uncertainty_network.network.nodes()
        dest_exists = _random_nodes[1] in _uncertainty_network.network.nodes()
        if src_exists and dest_exists:
            p = {"sender": _random_nodes[0], "receiver": _random_nodes[1], "amount": amount}
            _payments.append(p)
    # write payments to file
    ndjson.dump(_payments, open("data/payments.ndjson", "w"))
    return 0


# + Creation of a collection of N payments (src, rcv, amount)
create_payment_set(uncertainty_network, 200, 1000000)


In [ ]:
payment_set = ndjson.load(open("data/payments.ndjson", "r"))
logging.debug("Payments:\n%s", ndjson.dumps(payment_set))
logging.info("A total of {} payments.".format(len(payment_set)))

In [ ]:
print("===PICKHARDT PAY===")
sim_session.forget_information()
p = Payment(uncertainty_network, oracle_lightning_network, "B", "C", 300000, mu=0, base=0)
sim_session.pickhardt_pay(p.sender, p.receiver, p.total_amount, mu=0, base=0)
print("=====")
sim_session.forget_information()
p = Payment(uncertainty_network, oracle_lightning_network, "A", "E", 500000, mu=0, base=0)
sim_session.pickhardt_pay(p.sender, p.receiver, p.total_amount, mu=0, base=0)
print("=====")
sim_session.forget_information()
p = Payment(uncertainty_network, oracle_lightning_network, "A", "E", 300000, mu=0, base=0)
sim_session.pickhardt_pay(p.sender, p.receiver, p.total_amount, mu=0, base=0)

In [ ]:
print("===PAY===")
sim_session.forget_information()
p = Payment(uncertainty_network, oracle_lightning_network, "B", "C", 300000, mu=0, base=0)
sim_session.pay(p.sender, p.receiver, p.total_amount, mu=0, base=0)
print("=====")


In [ ]:
import ortools

# ortools.graph.DijkstraShortestPath(node_count=20, start_node=17, end_node=18, graph=dist, disconnected_distance=300)

In [ ]:
# Given a large random network find the shortest path from '0' to '5'
print(sim_session.dijkstra(oracle_lightning_network, s='B', t='C'))

### General descriptives of data set
sample payment pairs
sample success rate for payment to find out how large a sample should be to be stable
sample success rate for pickhardt payment to find out how large a sample should be to be stable
Decide on sample size

### Then

save successful payments/Attempts from sample to json
iterate over edges and find out "most frequent edges" in payment delivery
do flows share common channels (for n payments)?

build pay function with dijkstra/yen’s algorithm

## Questions
how does pay improve, when belief about channel liquidity is retained in an uncertainty network? Compared to not sharing this belief/nodes retain their info.
how does probabilistic payment improve, when belief about channel liquidity is retained in an uncertainty network? Compared to not sharing this belief/nodes retain their info.

Is the uncertainty network/stored belief like a load balancer for payments? Does load balancing the channels according to central belief work?

Two independent payments can have the same flow but a lower success rate, when compared to probabilistic payment of aggregate and using min cost flow solver for optimal split.

is maintenance of the uncertainty network more important than achieving an optimal split when using the min cost flow solver?